# Módulo 01 · pandas 3, dtypes y carga eficiente

**Curso de pandas intermedio → avanzado** · Dataset: *Olist Brazilian E-commerce* (muestras en `data/`)

## Objetivos
Al terminar este módulo podrás:

1. Explicar los dos cambios de pandas 3 que más rompen código viejo: **dtype `str` por defecto** y **Copy-on-Write**.
2. Auditar los dtypes y la memoria real de un DataFrame (`memory_usage(deep=True)`).
3. Cargar CSVs con el **dtype correcto desde el inicio** (`dtype=`, `parse_dates=`, `usecols=`).
4. Usar **dtypes nullable** (`Int64`, `Float64`, `boolean`) y entender `pd.NA` vs `NaN`.
5. Usar **`category`** (incluyendo categorías ordenadas) para ahorrar memoria y dar semántica.
6. Comparar el backend por defecto contra **PyArrow** y guardar en **Parquet** sin perder tipos.

## Estructura
| Sección | Tema |
|---|---|
| 0 | Setup |
| 1 | Conociendo el modelo de datos de Olist |
| 2 | Cambios clave de pandas 3 |
| 3 | Auditar dtypes y memoria |
| 4 | Leer con el dtype correcto |
| 5 | Dtypes nullable y `pd.NA` |
| 6 | `category` |
| 7 | Fechas y duraciones |
| 8 | Backend PyArrow y Parquet |
| 9 | Patrón: esquema declarativo de dtypes |
| ✍️ | Ejercicios (con autocomprobación) |
| ✅ | Soluciones |

> **Cómo usarlo:** ejecuta las celdas en orden. En cada ejercicio hay una celda `# TU CÓDIGO AQUÍ` seguida de una celda de **comprobación** con `assert`s. Intenta resolver antes de ver las soluciones al final.

## 0 · Setup

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import pyarrow as pa

DATA = Path("data")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

print("Python :", sys.version.split()[0])
print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("pyarrow:", pa.__version__)
assert int(pd.__version__.split(".")[0]) >= 3, "Este curso requiere pandas 3.x"


def pendiente(*valores) -> bool:
  # True si algún valor sigue siendo `...` (ejercicio sin resolver)
  return any(v is ... for v in valores)

In [ ]:
sorted(p.name for p in DATA.glob("*.csv"))

## 1 · Conociendo el modelo de datos de Olist

Olist es un marketplace brasileño. Las tablas se relacionan así (llaves entre paréntesis):

```
customers (customer_id) ──< orders (order_id) ──< order_items >── products (product_id) ── category_translation
                                   │                   │                                    (product_category_name)
                                   ├──< payments       └──> sellers (seller_id)
                                   └──< reviews
customers / sellers ──(zip_code_prefix)──> geolocation   (¡muchas filas por prefijo!)
```

Puntos importantes que usaremos durante todo el curso:

- Un **pedido** puede tener varios **ítems** (`order_item_id` = 1, 2, 3…), varios **pagos** (`payment_sequential`) y en algunos casos más de una **reseña**.
- `customer_id` es **por pedido**; la persona real se identifica con `customer_unique_id`.
- `geolocation` tiene **muchas coordenadas por prefijo postal**: unirla sin agregar primero multiplica filas.

Carguemos todo "a lo ingenuo" para ver qué nos da pandas por defecto:

In [ ]:
def nombre_corto(p: Path) -> str:
  # olist_orders_dataset.csv -> "orders"; product_category_name_translation.csv -> "category_translation"
  return (p.stem.replace("olist_", "").replace("_dataset", "")
                .replace("product_category_name_translation", "category_translation"))

raw = {nombre_corto(p): pd.read_csv(p) for p in sorted(DATA.glob("*.csv"))}

pd.DataFrame({
  "filas": {k: len(v) for k, v in raw.items()},
  "columnas": {k: v.shape[1] for k, v in raw.items()},
})

In [ ]:
raw["orders"].head()

## 2 · Cambios clave de pandas 3

### 2.1 El texto ya no es `object`: es `str`

En pandas 2 una columna de texto era `object` (un array de punteros a objetos Python). En pandas 3 el texto se infiere como el dtype **`str`**, respaldado por PyArrow cuando está instalado. Consecuencias:

- Menos memoria y operaciones `.str` más rápidas.
- Los faltantes en una columna `str` se representan como `NaN`.
- Código que hacía `df["col"].dtype == object` para detectar texto **deja de funcionar**. Usa `pd.api.types.is_string_dtype`.

In [ ]:
orders = raw["orders"]
orders.dtypes

In [ ]:
col = orders["order_status"]
print("dtype            :", col.dtype)
print("¿== object?      :", col.dtype == object)                        # ❌ patrón viejo
print("is_string_dtype  :", pd.api.types.is_string_dtype(col))          # ✅ patrón correcto
print("is_object_dtype  :", pd.api.types.is_object_dtype(col))

In [ ]:
# Seleccionar columnas de texto de forma robusta
orders.select_dtypes(include="str").columns.tolist()

### 2.2 Copy-on-Write (CoW) siempre activo

Con CoW, **cualquier objeto derivado de otro (un slice, una columna, un filtro) se comporta como copia**. Internamente pandas evita copiar hasta que alguien modifica algo, así que es eficiente.

Lo que esto significa en la práctica:

1. La **asignación encadenada** (`df[filtro]["col"] = valor`) **nunca** modifica el original. pandas emite un `ChainedAssignmentError`.
2. Modificar una columna extraída (`s = df["col"]; s.iloc[0] = ...`) **no** toca `df`.
3. El famoso `SettingWithCopyWarning` desaparece: ya no hay ambigüedad.

**Regla de oro:** para modificar, usa **un solo** `.loc[filas, columnas] = valor`, o crea columnas nuevas con `assign`.

In [ ]:
items = raw["order_items"].copy()

# ❌ Asignación encadenada: NO modifica `items` (verás un ChainedAssignmentError)
items[items["price"] > 1000]["freight_value"] = 0
print("Fletes en 0 tras encadenada:", (items["freight_value"] == 0).sum())

In [ ]:
# ✅ Una sola operación .loc
items.loc[items["price"] > 1000, "freight_value"] = 0
print("Fletes en 0 tras .loc:", (items["freight_value"] == 0).sum())

In [ ]:
# Una columna extraída es independiente del DataFrame
precios = items["price"]
precios.iloc[0] = -1
print("Serie extraída :", precios.iloc[0])
print("DataFrame      :", items["price"].iloc[0])

> 💡 **¿Y `inplace=True`?** Sigue existiendo en muchos métodos, pero no ahorra memoria con CoW y no encadena bien. En este curso usaremos el estilo `df = df.metodo(...)` y *method chaining* (Módulo 08).

## 3 · Auditar dtypes y memoria

`df.info()` reporta memoria **aproximada** a menos que pidas `memory_usage="deep"`. Para texto y `object` la diferencia es enorme.

In [ ]:
orders.info(memory_usage="deep")

In [ ]:
def perfil(df: pd.DataFrame) -> pd.DataFrame:
  # Resumen por columna: dtype, nulos, cardinalidad y memoria real.
  return pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "pct_nulos": df.isna().mean().mul(100).round(1),
    "unicos": df.nunique(),
    "mem_KB": df.memory_usage(deep=True, index=False).div(1024).round(1),
  }).sort_values("mem_KB", ascending=False)

perfil(orders)

Lee este perfil como un analista:

- **Fechas como `str`**: los 5 timestamps se leyeron como texto. No podemos restar fechas ni hacer `resample`.
- **`order_status` con muy pocos valores únicos**: candidato perfecto a `category`.
- **Nulos en fechas de entrega**: tienen sentido de negocio (pedidos cancelados o en tránsito), no son errores.

In [ ]:
perfil(raw["products"])

Dos trampas en `products`:

- `product_photos_qty` es `float64` aunque es un conteo entero. **¿Por qué?** Porque tiene nulos y NumPy no tiene enteros con faltantes; pandas "asciende" a float para poder usar `NaN`.
- Los nombres `..._lenght` tienen un typo heredado del dataset original. Es común renombrarlos al cargar.

In [ ]:
perfil(raw["sellers"])

Y una trampa silenciosa en `sellers`/`customers`/`geolocation`: **el código postal se leyó como entero.** Los CEP brasileños tienen 5 dígitos en el prefijo y algunos empiezan con `0` (São Paulo capital: `01000`–`05999`). Al convertirlos a `int` se pierde el cero inicial:

In [ ]:
raw["sellers"].loc[raw["sellers"]["seller_city"].eq("sao paulo"), "seller_zip_code_prefix"].head()

## 4 · Leer con el dtype correcto desde el inicio

Convertir después funciona, pero **declarar los tipos al leer** es más rápido, usa menos memoria pico y documenta tus supuestos. Los parámetros clave de `read_csv`:

| Parámetro | Para qué |
|---|---|
| `usecols` | Leer sólo las columnas que necesitas |
| `dtype` | Dict `{columna: dtype}` |
| `parse_dates` | Lista de columnas a convertir a `datetime64` |
| `converters` | Función por columna (más lento; úsalo poco) |
| `na_values` | Valores extra a tratar como faltantes |
| `engine="pyarrow"` | Parser multihilo en C++ |

In [ ]:
ORDER_DATES = [
  "order_purchase_timestamp", "order_approved_at",
  "order_delivered_carrier_date", "order_delivered_customer_date",
  "order_estimated_delivery_date",
]

orders = pd.read_csv(
  DATA / "olist_orders_dataset.csv",
  dtype={"order_status": "category"},
  parse_dates=ORDER_DATES,
)
perfil(orders)

Para el código postal leemos como texto y rellenamos con ceros a la izquierda con `str.zfill`:

In [ ]:
sellers = pd.read_csv(
  DATA / "olist_sellers_dataset.csv",
  dtype={"seller_zip_code_prefix": "str", "seller_state": "category"},
)
# Al leer como str el valor ya viene sin el cero ("3702"), así que normalizamos a 5 caracteres
sellers["seller_zip_code_prefix"] = sellers["seller_zip_code_prefix"].str.zfill(5)

sellers.loc[sellers["seller_city"].eq("sao paulo"), ["seller_zip_code_prefix", "seller_city"]].head()

`usecols` también acepta una función, útil cuando quieres "todas menos…":

In [ ]:
reviews_sin_texto = pd.read_csv(
  DATA / "olist_order_reviews_dataset.csv",
  usecols=lambda c: not c.startswith("review_comment"),
)
reviews_sin_texto.columns.tolist()

## 5 · Dtypes nullable y `pd.NA`

pandas tiene **dos familias** de dtypes:

| Familia | Ejemplos | Faltante | Comentario |
|---|---|---|---|
| NumPy | `int64`, `float64`, `bool` | `NaN` (sólo floats) | Los enteros con faltantes se convierten a float |
| Extension (nullable) | `Int64`, `Float64`, `boolean` | `pd.NA` | Enteros y booleanos que **sí** admiten faltantes |

> Observa la mayúscula: `int64` (NumPy) ≠ `Int64` (nullable).

In [ ]:
products = raw["products"]
fotos = products["product_photos_qty"]

print("Original :", fotos.dtype, "| valores:", fotos.dropna().unique()[:5])
fotos_int = fotos.astype("Int64")
print("Nullable :", fotos_int.dtype)
fotos_int[fotos_int.isna()].head(3)

`pd.NA` usa **lógica de tres valores** (como SQL). Una comparación con un faltante da faltante, no `False`:

In [ ]:
print("NaN == NaN   ->", np.nan == np.nan)
print("pd.NA == 1   ->", pd.NA == 1)
print("pd.NA | True ->", pd.NA | True)    # True sin importar el faltante
print("pd.NA & True ->", pd.NA & True)    # desconocido

In [ ]:
# Un filtro booleano nullable trata NA como False al indexar
mas_de_3_fotos = fotos_int > 3
print(mas_de_3_fotos.dtype)
print("NA en la máscara:", mas_de_3_fotos.isna().sum())
print("Filas filtradas :", len(products[mas_de_3_fotos]))

`convert_dtypes()` elige automáticamente el mejor dtype nullable para cada columna. Muy útil como primer paso exploratorio:

In [ ]:
products.convert_dtypes().dtypes

## 6 · `category`

Una columna `category` guarda **un diccionario de valores únicos** + **códigos enteros** por fila. Brilla cuando hay **pocos valores repetidos muchas veces** (estados, estatus, métodos de pago).

In [ ]:
pagos = raw["order_payments"]

tipo_str = pagos["payment_type"]
tipo_cat = tipo_str.astype("category")

print("str     :", tipo_str.memory_usage(deep=True), "bytes")
print("category:", tipo_cat.memory_usage(deep=True), "bytes")
print()
print("Categorías:", list(tipo_cat.cat.categories))
print("Códigos   :", tipo_cat.cat.codes.head().tolist())

### Categorías ordenadas

Cuando el orden importa (calificaciones, tallas, niveles), declara un `CategoricalDtype` ordenado. Ganas comparaciones `<`, `>`, `min`, `max` y orden correcto en gráficos y `groupby`.

In [ ]:
score_dtype = pd.CategoricalDtype(categories=[1, 2, 3, 4, 5], ordered=True)

reviews = pd.read_csv(DATA / "olist_order_reviews_dataset.csv",
                      dtype={"review_score": score_dtype})

print(reviews["review_score"].dtype)
print("Mínimo:", reviews["review_score"].min())
print("Reseñas ≤ 2:", (reviews["review_score"] <= 2).sum())

### Categorías no observadas

Si una categoría declarada no aparece en los datos, `value_counts` y `groupby` pueden mostrarla igual. En `groupby`, `observed=True` es el comportamiento por defecto en pandas 3 (sólo combinaciones que existen).

In [ ]:
estados_dtype = pd.CategoricalDtype(["delivered", "shipped", "invoiced", "processing",
                                     "approved", "created", "unavailable", "canceled"])
st = orders["order_status"].astype(estados_dtype)

st.value_counts()            # incluye categorías con 0

In [ ]:
st.value_counts().loc[lambda s: s > 0]

> ⚠️ **Cuándo NO usar `category`:** columnas casi únicas (`order_id`, textos libres). El diccionario sería tan grande como los datos y pagarías el costo de los códigos encima.

## 7 · Fechas y duraciones

Con las fechas como `datetime64` podemos restar, comparar y extraer componentes con el accessor `.dt`. Fíjate en la **resolución** que reporta pandas 3 (`[us]` = microsegundos).

In [ ]:
orders[ORDER_DATES].dtypes

In [ ]:
entregas = orders.assign(
  dias_entrega=(orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]).dt.days,
  dias_vs_estimado=(orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]).dt.days,
  llego_tarde=lambda d: d["dias_vs_estimado"] > 0,
  mes_compra=orders["order_purchase_timestamp"].dt.to_period("M"),
  dia_semana=orders["order_purchase_timestamp"].dt.day_name(),
)
entregas[["order_status", "dias_entrega", "dias_vs_estimado", "llego_tarde", "mes_compra", "dia_semana"]].head()

In [ ]:
entregas[["dias_entrega", "dias_vs_estimado"]].dtypes

¿Por qué `dias_entrega` es `float64`? Porque los pedidos no entregados tienen `NaT` y `.dt.days` produce `NaN`. Si quieres enteros con faltantes: `.astype("Int64")`.

In [ ]:
entregas["dias_entrega"].astype("Int64").describe()

## 8 · Backend PyArrow y Parquet

`dtype_backend="pyarrow"` hace que **todas** las columnas usen tipos Arrow (`int64[pyarrow]`, `timestamp[s][pyarrow]`, …), todos nullable. Combinado con `engine="pyarrow"` el parseo es multihilo.

In [ ]:
orders_arrow = pd.read_csv(DATA / "olist_orders_dataset.csv",
                           engine="pyarrow", dtype_backend="pyarrow")
orders_arrow.dtypes

In [ ]:
def mem_kb(df):
  return round(df.memory_usage(deep=True).sum() / 1024, 1)

pd.Series({
  "ingenuo (todo str)": mem_kb(raw["orders"]),
  "dtypes declarados": mem_kb(orders),
  "backend pyarrow": mem_kb(orders_arrow),
}, name="KB")

> Con 300 filas las diferencias son pequeñas; con el dataset completo (~100 mil pedidos) o tablas de millones de filas se vuelven decisivas. Lo importante es el **hábito**.

### Parquet: guarda los tipos junto con los datos

El CSV es texto: al releerlo pierdes todos los dtypes. Parquet es columnar, comprimido y **conserva el esquema** (categorías, fechas, nullable).

In [ ]:
CACHE = Path("cache")
CACHE.mkdir(exist_ok=True)

orders.to_parquet(CACHE / "orders.parquet", index=False)
orders.to_csv(CACHE / "orders.csv", index=False)

desde_parquet = pd.read_parquet(CACHE / "orders.parquet")
desde_csv = pd.read_csv(CACHE / "orders.csv")

pd.DataFrame({"original": orders.dtypes.astype(str),
              "parquet": desde_parquet.dtypes.astype(str),
              "csv": desde_csv.dtypes.astype(str)})

## 9 · Patrón: esquema declarativo de dtypes

Juntamos todo lo aprendido en un **esquema declarativo**: un diccionario que dice cómo leer cada tabla. Es un patrón útil en proyectos reales para tener los tipos en un solo lugar.

> En este curso **no** usaremos un módulo externo: cada notebook leerá sus tablas con `pd.read_csv` y declarará ahí mismo los dtypes que necesite. Así practicas el tipado en cada módulo y todo queda en pandas puro.

In [ ]:
SCHEMA = {
  "orders": dict(
    file="olist_orders_dataset.csv",
    dtype={"order_status": "category"},
    parse_dates=ORDER_DATES,
  ),
  "order_items": dict(
    file="olist_order_items_dataset.csv",
    dtype={"order_item_id": "int8"},
    parse_dates=["shipping_limit_date"],
  ),
  "order_payments": dict(
    file="olist_order_payments_dataset.csv",
    dtype={"payment_type": "category", "payment_sequential": "int8",
           "payment_installments": "int8"},
  ),
  "order_reviews": dict(
    file="olist_order_reviews_dataset.csv",
    dtype={"review_score": score_dtype},
    parse_dates=["review_creation_date", "review_answer_timestamp"],
  ),
  "customers": dict(
    file="olist_customers_dataset.csv",
    dtype={"customer_zip_code_prefix": "str", "customer_state": "category"},
  ),
  "sellers": dict(
    file="olist_sellers_dataset.csv",
    dtype={"seller_zip_code_prefix": "str", "seller_state": "category"},
  ),
  "products": dict(
    file="olist_products_dataset.csv",
    dtype={"product_category_name": "category"},
  ),
  "geolocation": dict(
    file="olist_geolocation_dataset.csv",
    dtype={"geolocation_zip_code_prefix": "str", "geolocation_state": "category"},
  ),
  "category_translation": dict(
    file="product_category_name_translation.csv",
  ),
}

ZIP_COLS = ["customer_zip_code_prefix", "seller_zip_code_prefix", "geolocation_zip_code_prefix"]
PRODUCT_RENAME = {"product_name_lenght": "product_name_length",
                  "product_description_lenght": "product_description_length"}


def load_table(name: str, data_dir: Path = DATA) -> pd.DataFrame:
  spec = dict(SCHEMA[name])
  df = pd.read_csv(data_dir / spec.pop("file"), **spec)
  for c in df.columns.intersection(ZIP_COLS):
    df[c] = df[c].str.zfill(5)
  if name == "products":
    df = df.rename(columns=PRODUCT_RENAME)
    cols_int = df.columns.drop(["product_id", "product_category_name"])
    df[cols_int] = df[cols_int].astype("Int64")
  return df


def load_all(data_dir: Path = DATA) -> dict[str, pd.DataFrame]:
  return {name: load_table(name, data_dir) for name in SCHEMA}


olist = load_all()
pd.DataFrame({
  "filas": {k: len(v) for k, v in olist.items()},
  "KB_ingenuo": {k: mem_kb(raw[k]) for k in olist},
  "KB_tipado": {k: mem_kb(v) for k, v in olist.items()},
}).assign(ahorro_pct=lambda d: (1 - d["KB_tipado"] / d["KB_ingenuo"]).mul(100).round(1))

In [ ]:
olist["products"].dtypes

---
# ✍️ Ejercicios

Cada ejercicio tiene una celda para tu código y otra de **comprobación**. Si la comprobación imprime ✅, vas bien.

Recordatorio de variables útiles ya definidas: `DATA`, `raw`, `perfil()`, `mem_kb()`, `score_dtype`.

### Ejercicio 1 · Reseñas bien tipadas ⭐

Lee `olist_order_reviews_dataset.csv` en un DataFrame llamado `rev` de forma que:

1. `review_score` sea una **categoría ordenada** 1 < 2 < 3 < 4 < 5.
2. `review_creation_date` y `review_answer_timestamp` sean `datetime64`.
3. Exista una columna nueva `horas_respuesta` = horas (float) entre la creación de la reseña y la respuesta.
4. Exista una columna booleana `tiene_comentario` que sea `True` si hay `review_comment_message`.

Después responde: ¿cuál es la **mediana** de `horas_respuesta` para reseñas con score ≤ 2? Guárdala en `mediana_negativas`.

In [ ]:
# TU CÓDIGO AQUÍ
rev = ...
mediana_negativas = ...

In [ ]:
# Comprobación
if pendiente(rev, mediana_negativas):
  print("⏳ Ejercicio 1 pendiente: resuélvelo en la celda de arriba")
else:
  assert isinstance(rev, pd.DataFrame), "rev debe ser un DataFrame"
  assert isinstance(rev["review_score"].dtype, pd.CategoricalDtype) and rev["review_score"].cat.ordered
  assert pd.api.types.is_datetime64_any_dtype(rev["review_creation_date"])
  assert pd.api.types.is_datetime64_any_dtype(rev["review_answer_timestamp"])
  assert {"horas_respuesta", "tiene_comentario"} <= set(rev.columns)
  assert rev["tiene_comentario"].sum() == raw["order_reviews"]["review_comment_message"].notna().sum()
  assert np.isclose(mediana_negativas, 31.6574, atol=0.01), f"Revisa tu cálculo: {mediana_negativas}"
  print("✅ Ejercicio 1 correcto")

### Ejercicio 2 · Códigos postales rotos ⭐

1. Usando `raw["customers"]` (leído de forma ingenua), cuenta cuántos clientes tienen un código postal que **perdió su cero inicial** (es decir, que al convertirse a texto tiene menos de 5 dígitos). Guárdalo en `n_rotos`.
2. Crea `cust`, una copia de `raw["customers"]` donde `customer_zip_code_prefix` sea texto de exactamente 5 caracteres.
3. ¿Qué **estados** (`customer_state`) tienen clientes con prefijo que empieza con `"0"`? Guárdalos como lista ordenada en `estados_cero`.

In [ ]:
# TU CÓDIGO AQUÍ
n_rotos = ...
cust = ...
estados_cero = ...

In [ ]:
# Comprobación
if pendiente(n_rotos, cust, estados_cero):
  print("⏳ Ejercicio 2 pendiente: resuélvelo en la celda de arriba")
else:
  assert n_rotos == 78, f"n_rotos = {n_rotos}"
  assert cust["customer_zip_code_prefix"].str.len().eq(5).all()
  assert pd.api.types.is_string_dtype(cust["customer_zip_code_prefix"])
  assert raw["customers"]["customer_zip_code_prefix"].dtype == "int64", "¡No modifiques raw! Usa .copy() o assign"
  assert estados_cero == ["SP"], f"estados_cero = {estados_cero}"
  print("✅ Ejercicio 2 correcto")

### Ejercicio 3 · Arregla el código (Copy-on-Write) ⭐⭐

Un compañero escribió este código pensado para pandas 1.x. Quiere marcar con `"sin_categoria"` los productos sin categoría y poner en `0` las fotos faltantes. **No funciona en pandas 3.** Reescríbelo para que sí funcione, sin modificar `raw["products"]`, y deja el resultado en `prod_fix`.

```python
prod_fix = raw["products"]
prod_fix[prod_fix["product_category_name"].isna()]["product_category_name"] = "sin_categoria"
prod_fix["product_photos_qty"][prod_fix["product_photos_qty"].isna()] = 0
```

**Extra:** después de arreglarlo, convierte `product_photos_qty` a `Int64` (¿sigue haciendo falta un dtype nullable?).

In [ ]:
# TU CÓDIGO AQUÍ
prod_fix = ...

In [ ]:
# Comprobación
if pendiente(prod_fix):
  print("⏳ Ejercicio 3 pendiente: resuélvelo en la celda de arriba")
else:
  assert prod_fix["product_category_name"].isna().sum() == 0
  assert (prod_fix["product_category_name"] == "sin_categoria").sum() == 6
  assert prod_fix["product_photos_qty"].isna().sum() == 0
  assert str(prod_fix["product_photos_qty"].dtype) == "Int64"
  assert raw["products"]["product_category_name"].isna().sum() == 6, "¡Modificaste raw!"
  print("✅ Ejercicio 3 correcto")

### Ejercicio 4 · Categorías sin traducción ⭐⭐

1. Carga `products` con `product_category_name` como `category`.
2. Usando **sólo operaciones de categorías** (sin `merge`), reemplaza las categorías en portugués por su nombre en inglés usando `product_category_name_translation.csv`. Pista: `rename_categories` acepta un diccionario.
3. Las categorías que **no tengan traducción** deben quedarse con su nombre original. Guárdalas en la lista ordenada `sin_traduccion`.
4. El resultado final en `prod_en` con la columna `product_category_name` traducida y aún de tipo `category`.

In [ ]:
# TU CÓDIGO AQUÍ
prod_en = ...
sin_traduccion = ...

In [ ]:
# Comprobación
if pendiente(prod_en, sin_traduccion):
  print("⏳ Ejercicio 4 pendiente: resuélvelo en la celda de arriba")
else:
  assert isinstance(prod_en["product_category_name"].dtype, pd.CategoricalDtype)
  assert sin_traduccion == [], f"sin_traduccion = {sin_traduccion}"
  assert "health_beauty" in prod_en["product_category_name"].cat.categories
  assert (prod_en["product_category_name"] == "health_beauty").sum() == 27
  print("✅ Ejercicio 4 correcto")

### Ejercicio 5 · Reto: benchmark de memoria en `geolocation` ⭐⭐⭐

`geolocation` es la tabla más grande y repetitiva. Construye un DataFrame `bench` con índice `["ingenuo", "categorias", "pyarrow"]` y una columna `KB` con la memoria total (deep) de:

- **ingenuo**: `pd.read_csv` sin argumentos.
- **categorias**: `geolocation_city` y `geolocation_state` como `category`, código postal como texto de 5 caracteres.
- **pyarrow**: `engine="pyarrow", dtype_backend="pyarrow"`.

Agrega una columna `vs_ingenuo` con el porcentaje respecto a la versión ingenua (100 = igual).

**Bonus:** las ciudades de `geolocation` traen acentos (`"são paulo"`) pero en `customers` no (`"sao paulo"`). Crea `ciudad_norm` sin acentos usando `.str.normalize("NFKD")` y luego eliminando los caracteres combinantes. ¿Cuántas ciudades únicas había antes y cuántas después? (`n_antes`, `n_despues`)

In [ ]:
# TU CÓDIGO AQUÍ
bench = ...
n_antes, n_despues = ..., ...

In [ ]:
# Comprobación
if pendiente(bench, n_antes, n_despues):
  print("⏳ Ejercicio 5 pendiente: resuélvelo en la celda de arriba")
else:
  assert list(bench.index) == ["ingenuo", "categorias", "pyarrow"]
  assert {"KB", "vs_ingenuo"} <= set(bench.columns)
  assert bench.loc["categorias", "KB"] < bench.loc["ingenuo", "KB"]
  assert np.isclose(bench.loc["ingenuo", "vs_ingenuo"], 100)
  assert (n_antes, n_despues) == (192, 149), (n_antes, n_despues)
  print("✅ Ejercicio 5 correcto")
  display(bench)

---
# ✅ Soluciones

> Intenta los ejercicios antes de mirar aquí. Las soluciones sobrescriben las variables de los ejercicios, así que al ejecutarlas también pasan las comprobaciones de arriba.

### Solución 1

In [ ]:
rev = (
  pd.read_csv(
    DATA / "olist_order_reviews_dataset.csv",
    dtype={"review_score": score_dtype},
    parse_dates=["review_creation_date", "review_answer_timestamp"],
  )
  .assign(
    horas_respuesta=lambda d: (d["review_answer_timestamp"] - d["review_creation_date"])
                               .dt.total_seconds() / 3600,
    tiene_comentario=lambda d: d["review_comment_message"].notna(),
  )
)
mediana_negativas = rev.loc[rev["review_score"] <= 2, "horas_respuesta"].median()
round(mediana_negativas, 2)

*Nota:* `.dt.days` truncaría a días completos; `.dt.total_seconds()` conserva la fracción. Como `review_score` es categoría **ordenada**, `<= 2` funciona directamente.

### Solución 2

In [ ]:
zip_txt = raw["customers"]["customer_zip_code_prefix"].astype("str")
n_rotos = zip_txt.str.len().lt(5).sum()

cust = raw["customers"].assign(customer_zip_code_prefix=zip_txt.str.zfill(5))

estados_cero = sorted(
  cust.loc[cust["customer_zip_code_prefix"].str.startswith("0"), "customer_state"].unique()
)
n_rotos, estados_cero

*Nota:* `assign` devuelve un DataFrame nuevo, así que `raw` queda intacto sin necesidad de `.copy()` explícito. Que todos sean `SP` tiene sentido: los CEP `0xxxx` corresponden a la región metropolitana de São Paulo.

### Solución 3

In [ ]:
prod_fix = raw["products"].copy()          # copia explícita para no tocar raw
prod_fix.loc[prod_fix["product_category_name"].isna(), "product_category_name"] = "sin_categoria"
prod_fix.loc[prod_fix["product_photos_qty"].isna(), "product_photos_qty"] = 0
prod_fix["product_photos_qty"] = prod_fix["product_photos_qty"].astype("Int64")

# Alternativa idiomática, sin mutar nada:
prod_fix_alt = raw["products"].fillna({"product_category_name": "sin_categoria",
                                       "product_photos_qty": 0}) \
                              .astype({"product_photos_qty": "Int64"})
assert prod_fix.equals(prod_fix_alt)
prod_fix["product_photos_qty"].dtype

*Sobre el extra:* tras rellenar ya no hay faltantes, así que `int64` (NumPy) también serviría. Aun así `Int64` es buena elección si el DataFrame seguirá transformándose (un `merge` posterior podría reintroducir faltantes y con NumPy volverías a `float`).

### Solución 4

In [ ]:
traduccion = pd.read_csv(DATA / "product_category_name_translation.csv")
mapa = dict(zip(traduccion["product_category_name"], traduccion["product_category_name_english"]))

prod_en = pd.read_csv(DATA / "olist_products_dataset.csv",
                      dtype={"product_category_name": "category"})

cats = prod_en["product_category_name"].cat.categories
sin_traduccion = sorted(set(cats) - set(mapa))

prod_en["product_category_name"] = prod_en["product_category_name"].cat.rename_categories(
  {c: mapa[c] for c in cats if c in mapa}
)
prod_en["product_category_name"].value_counts().head()

*Por qué así:* `rename_categories` sólo toca el **diccionario** de categorías (unas decenas de valores), no las filas. En una tabla de millones de filas es muchísimo más barato que un `merge` o un `.map()` fila por fila. En esta muestra todas las categorías tienen traducción; en el dataset completo hay un par que no (por ejemplo `pc_gamer`), y este código las deja intactas.

### Solución 5

In [ ]:
geo_file = DATA / "olist_geolocation_dataset.csv"

geo_ing = pd.read_csv(geo_file)
geo_cat = pd.read_csv(
  geo_file,
  dtype={"geolocation_zip_code_prefix": "str",
         "geolocation_city": "category",
         "geolocation_state": "category"},
).assign(geolocation_zip_code_prefix=lambda d: d["geolocation_zip_code_prefix"].str.zfill(5))
geo_arrow = pd.read_csv(geo_file, engine="pyarrow", dtype_backend="pyarrow")

bench = (
  pd.DataFrame({"KB": [mem_kb(geo_ing), mem_kb(geo_cat), mem_kb(geo_arrow)]},
               index=["ingenuo", "categorias", "pyarrow"])
  .assign(vs_ingenuo=lambda d: (d["KB"] / d.loc["ingenuo", "KB"] * 100).round(1))
)

# Bonus: quitar acentos
ciudad = geo_ing["geolocation_city"]
ciudad_norm = (ciudad.str.normalize("NFKD")
                     .str.encode("ascii", errors="ignore")
                     .str.decode("ascii"))
n_antes, n_despues = ciudad.nunique(), ciudad_norm.nunique()
print(f"Ciudades únicas: {n_antes} → {n_despues}")
bench

*Lectura del resultado:* la versión con categorías gana porque ciudades y estados se repiten muchísimo. El backend pyarrow ahorra frente al ingenuo en los números pero no comprime repeticiones. **Las técnicas se combinan**: puedes tener columnas Arrow y aun así declarar `category` donde convenga.

El bonus anticipa el Módulo 07 (texto y datos sucios): sin normalizar, "são paulo" y "sao paulo" serían ciudades distintas y cualquier `merge` por ciudad fallaría.

---
## Resumen del módulo

- En pandas 3 el texto es `str`; detecta texto con `pd.api.types.is_string_dtype`, no con `== object`.
- Copy-on-Write: **un solo `.loc`** para modificar; nada de asignación encadenada.
- Declara tipos al leer: `dtype=`, `parse_dates=`, `usecols=`. Códigos postales y IDs con ceros iniciales → **texto**.
- `Int64`/`Float64`/`boolean` admiten `pd.NA` (lógica de tres valores).
- `category` para baja cardinalidad; **ordenada** cuando el orden tiene significado.
- Guarda resultados intermedios en **Parquet** para conservar el esquema.

**Siguiente:** Módulo 02 · Indexación avanzada y `MultiIndex`. Cada notebook cargará sus tablas directamente con `pd.read_csv`, declarando los dtypes que use.